# GNN-Based BERT for Understanding Context from Music
### Task 3: Multi-Modal Context Understanding Demo
This interactive notebook demonstrates the end-to-end pipeline of our hybrid GNN-BERT fusion architecture.

In [ ]:
!pip install -q torch_geometric transformers librosa soundfile

import torch
from torch_geometric.loader import DataLoader

# Import our custom modules
from main import GTZANDemoDataset
from src.train import train_fusion_model
from src.evaluate import evaluate_fusion_model

## 1. Dataset Initialization & Graph Construction
We dynamically load the GTZAN dataset, convert audio into PyTorch Geometric segment graphs, and tokenize proxy textual labels using HuggingFace BERT.

In [ ]:
DATA_DIR = "Data/genres_original"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {device}")
print("Loading Full Dataset...")

dataset = GTZANDemoDataset(DATA_DIR, max_files_per_genre=100)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

## 2. Model Training (Cross-Attention Fusion)
Here we train the model for 25 epochs. The GraphSAGE output is fused with the BERT embeddings via cross-attention.

In [ ]:
print("\nStarting 25-Epoch Training...")
trained_model = train_fusion_model(
    train_loader=train_loader, 
    num_classes=len(dataset.genres), 
    epochs=25, 
    device=device
)

## 3. Model Evaluation
Generating ranking and classification metrics based on our final converged model.

In [ ]:
print("\nEvaluating Model Performance...")
metrics = evaluate_fusion_model(trained_model, train_loader, device=device)

## 4. Visualizations
Visualizing the topological structure of our fused latent representations using t-SNE.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np

trained_model.eval()
all_z = []
all_labels = []

with torch.no_grad():
    for batch_graphs, input_ids, attention_mask, labels in train_loader:
        batch_graphs = batch_graphs.to(device)
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        
        _, z = trained_model(batch_graphs.x, batch_graphs.edge_index, batch_graphs.batch, input_ids, attention_mask)
        all_z.append(z.cpu().numpy())
        all_labels.extend(torch.argmax(labels, dim=1).cpu().numpy())

all_z = np.vstack(all_z)
perplexity = min(30, len(all_z) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
z_tsne = tsne.fit_transform(all_z)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(z_tsne[:, 0], z_tsne[:, 1], c=all_labels, cmap='tab10', s=50, alpha=0.8)
cbar = plt.colorbar(scatter)
cbar.set_label('Genre ID')
plt.title("t-SNE Visualization of GNN-BERT Latent Space")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()